# Lesson 2a Demo: API Basics and Prompting Techniques

1. **Chat Completions vs Responses API** — two API styles, same models, different tradeoffs
2. **System prompts & temperature** — steering model behavior
3. **Structured data extraction** — getting reliable JSON from LLMs
4. **Few-shot learning** — teaching by example
5. **Chain-of-thought prompting** — step-by-step reasoning
6. **Needle in a haystack** — long-context retrieval

Run cells in order.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


## 1) Chat Completions vs Responses API

In Lesson 1 we used the **Chat Completions API** (`client.chat.completions.create`). It's stateless: every call is a fresh start. The model only sees what you send in that request.

The **Responses API** (`client.responses.create`) can manage state **server-side** — if you chain calls with `previous_response_id`, the model remembers prior turns without you resending them.

| | Chat Completions | Responses API |
|---|---|---|
| **Mental model** | Single turn: messages in, message out | Multi-step: the model can take *actions* before responding |
| **Built-in tools** | None — you implement tool calls yourself | `web_search`, `file_search`, `code_interpreter` built in |
| **State** | You manage conversation history | Server can manage it (with `previous_response_id`) |
| **Context management** | You truncate/summarize manually | Server-side compaction available |

### Demo: no memory vs free memory

We'll ask the model a name, then ask it back — first with Chat Completions (no memory), then with Responses API (memory for free).

In [ ]:
# --- Chat Completions: NO memory ---
print("=== Chat Completions API (stateless) ===\n")

print('User: "My name is Alice."')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}],
)
print(f"Assistant: {r1.choices[0].message.content}\n")

print('User: "What is my name?"')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print(f"Assistant: {r2.choices[0].message.content}")
print("  ^ It has no idea — each call is independent.\n")

# --- Responses API: memory for free ---
print("=== Responses API (server-side state) ===\n")

resp1 = client.responses.create(
    model=MODEL,
    input="My name is Alice.",
)
print("Turn 1:", resp1.output_text)

resp2 = client.responses.create(
    model=MODEL,
    input="What is my name?",
    previous_response_id=resp1.id,  # <-- this is all it takes
)
print("Turn 2:", resp2.output_text)
print("  ^ It remembers — the server kept the context.")

### Built-in tools: web search

The Responses API also has **built-in tools** — the model can autonomously search the web, run code, or read files *within a single API call*, before returning a final answer.

Here we give it `web_search`. The model decides to search, reads the results, and synthesizes an answer — all in one call. No tool-calling code on our side.

In [ ]:
try:
    resp = client.responses.create(
        model=MODEL,
        tools=[{"type": "web_search"}],
        tool_choice="auto",
        input="Find one positive AI news story from this week. Include source.",
    )
    print(resp.output_text)
except Exception as e:
    print("Web search demo unavailable in this project or region.")
    print(type(e).__name__, e)

### System prompts

The `system` role lets you steer *how* the model responds — its tone, format, persona — without changing the user's question. Same input, different behavior.

In [ ]:
question = "What is Software 3.0?"

system_prompts = [
    None,  # no system message
    "You are a sarcastic comedian. Keep it short.",
    "You are a formal academic. Respond in exactly one sentence.",
    "You are a 5-year-old explaining things to another 5-year-old.",
]

for sp in system_prompts:
    messages = []
    if sp:
        messages.append({"role": "system", "content": sp})
    messages.append({"role": "user", "content": question})

    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.7)

    label = f'"{sp}"' if sp else "(no system message)"
    print(f"System: {label}")
    print(f"  → {r.choices[0].message.content}\n")

## Temperature

Temperature controls **randomness** in the model's output. At `0` the model always picks the most likely next token. At higher values it samples more freely, producing more varied — and sometimes more creative — responses.

**Surprise:** even at `temperature=0`, you may see slightly different outputs across runs. This is because GPU floating-point arithmetic isn't perfectly deterministic — different hardware, different request routing, tiny rounding differences. OpenAI offers a `seed` parameter to improve reproducibility, but doesn't guarantee it.

Same prompt, three temperatures, three runs each.

In [ ]:
prompt = "Explain Software 3.0 in one sentence."

print("=== temperature=0 (no seed) ===")
for run in range(1, 6):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=0, seed=42 ===")
for run in range(1, 6):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        seed=42,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=0.7 ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=1.5 ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.5,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

## Structured Data Extraction

LLMs can extract structured data from unstructured text — but getting **valid, parseable output** is harder than it looks. We ask the model to return JSON with named entities. Then we compare:

1. A strong model (`gpt-5.2`) — reliably follows the format
2. A weak model (`gpt-4.1-nano`, `temperature=1`) — may produce broken JSON, miss entities, or hallucinate fields

This is a preview of a recurring theme: **the gap between "usually works" and "always works" is where engineering lives.**

In [ ]:
import json

NER_PROMPT = """Extract all named entities from the following text and return them as JSON.

Classify each entity as one of: PERSON, ORGANIZATION, LOCATION, DATE, or MISC.

Text: "Apple CEO Tim Cook announced a new partnership with Microsoft in Seattle last Tuesday. The deal was brokered by Goldman Sachs."

Return only valid JSON in this format:
{
  "entities": [
    {"text": "...", "type": "...", "start": <char_offset>}
  ]
}"""

# --- Strong model, temp=0 ---
print("=== gpt-5.2 (strong model, temperature=0) ===\n")
r_strong = client.chat.completions.create(
    model="gpt-5.2",
    messages=[{"role": "user", "content": NER_PROMPT}],
    temperature=0,
)
raw_strong = r_strong.choices[0].message.content
print(raw_strong)

try:
    parsed = json.loads(raw_strong)
    print(f"\n✓ Valid JSON — {len(parsed['entities'])} entities extracted")
except json.JSONDecodeError as e:
    print(f"\n✗ Invalid JSON: {e}")

# --- Strong model, temp=1 ---
print("\n\n=== gpt-5.2 (strong model, temperature=1) — 3 runs ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model="gpt-5.2",
        messages=[{"role": "user", "content": NER_PROMPT}],
        temperature=1,
    )
    raw = r.choices[0].message.content
    print(f"\n--- Run {run} ---")
    print(raw[:500])
    try:
        parsed = json.loads(raw)
        print(f"✓ Valid JSON — {len(parsed['entities'])} entities")
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON: {e}")

# --- Weak model, temp=1 ---
print("\n\n=== gpt-4.1-nano (weak model, temperature=1) — 3 runs ===")
for run in range(1, 4):
    r_weak = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": NER_PROMPT}],
        temperature=1,
    )
    raw_weak = r_weak.choices[0].message.content
    print(f"\n--- Run {run} ---")
    print(raw_weak[:500])
    try:
        parsed = json.loads(raw_weak)
        print(f"✓ Valid JSON — {len(parsed['entities'])} entities")
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON: {e}")

## Few-Shot Learning

You can teach the model a new task **by example** — no fine-tuning needed. Just include a few input/output pairs in the prompt, and the model picks up the pattern.

This is called **few-shot prompting** (vs. **zero-shot**, where you give no examples). It's especially useful for tasks where the desired format or logic is hard to describe in words but easy to show.

Below we compare zero-shot vs few-shot on a sentiment classification task.

In [ ]:
test_review = "Oh wow, another laptop that mass-produces heat! At least the keyboard is nice while my desk slowly melts."

# --- Zero-shot: just ask ---
print("=== Zero-shot ===\n")
r_zero = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"Classify the sentiment of this product review as POSITIVE, NEGATIVE, or MIXED. Return only the label.\n\nReview: \"{test_review}\""},
    ],
    temperature=0,
)
print(f"Review: \"{test_review}\"")
print(f"Model:  {r_zero.choices[0].message.content}\n")

# --- Few-shot: teach by example ---
print("=== Few-shot (3 examples) ===\n")
r_few = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify product review sentiment. Reply with exactly one label: POSITIVE, NEGATIVE, or MIXED."},
        {"role": "user", "content": "Review: \"Sure, it 'works' — if you enjoy watching a loading spinner all day. Beautiful design though.\""},
        {"role": "assistant", "content": "MIXED"},
        {"role": "user", "content": "Review: \"Best paperweight I've ever bought! Totally worth $1,200 for something that crashes every hour.\""},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "Review: \"Camera is amazing but the software is buggy and slow.\""},
        {"role": "assistant", "content": "MIXED"},
        {"role": "user", "content": f"Review: \"{test_review}\""},
    ],
    temperature=0,
)
print(f"Review: \"{test_review}\"")
print(f"Model:  {r_few.choices[0].message.content}")
print("\nThe few-shot examples teach the model to recognize sarcasm — a genuine compliment mixed with sarcastic complaints → MIXED.")

## Chain-of-Thought Prompting

Some problems are hard to get right in one shot — especially reasoning, math, and multi-step logic. **Chain-of-thought (CoT)** prompting asks the model to show its reasoning step by step before giving a final answer.

Why does this help? The model's "thinking" tokens act as intermediate computation — each token can attend to the reasoning so far, reducing the chance of skipping a step.

Below we compare a direct answer vs chain-of-thought on a tricky word problem.

In [ ]:
problem = """A train leaves Station A at 9:00 AM heading east at 60 mph. Another train leaves Station B (which is 300 miles east of A) at 10:00 AM heading west at 90 mph. At what time do the trains meet?"""

# --- Direct answer ---
print("=== Direct (no CoT) ===\n")
r_direct = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + "\n\nAnswer with just the time."},
    ],
    temperature=0,
)
print(f"Problem: {problem.strip()}")
print(f"Answer:  {r_direct.choices[0].message.content}")
print("  (correct answer: 11:36 AM)\n")

# --- Chain-of-thought ---
print("=== Chain-of-Thought ===\n")
r_cot = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + "\n\nThink step by step, then give your final answer."},
    ],
    temperature=0,
)
print(f"Answer:\n{r_cot.choices[0].message.content}")

# --- A harder problem where CoT matters more ---
hard_problem = """A car drives 120 km in 1.5 hours, then stops for 30 minutes, then drives 80 km in 1 hour. What is the average speed for the entire trip (including the stop)?"""

print("\n\n=== Harder problem — Direct ===\n")
r2_direct = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": hard_problem + "\n\nAnswer with just the number."},
    ],
    temperature=0,
)
print(f"Answer: {r2_direct.choices[0].message.content}")
print("  (correct answer: ~66.67 km/h)")

print("\n=== Harder problem — Chain-of-Thought ===\n")
r2_cot = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": hard_problem + "\n\nThink step by step, then give your final answer."},
    ],
    temperature=0,
)
print(f"Answer:\n{r2_cot.choices[0].message.content}")

### Structured Chain-of-Thought

Instead of just saying "think step by step", you can **prescribe the output format** — forcing the model to separate reasoning from the final answer. This makes outputs easier to parse programmatically and reduces the chance of the model skipping steps.

In [ ]:
tennis_problem = """Roger has 5 tennis balls. 
He buys 2 more cans of tennis balls. 
Each can has 3 tennis balls.

How many tennis balls does he have now?"""

# --- Direct: just ask for the answer ---
print("=== Direct (no CoT) ===\n")
r_direct = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": tennis_problem + "\n\nAnswer with just the number."}],
    temperature=0,
)
print(f"Answer: {r_direct.choices[0].message.content}\n")

# --- Structured CoT: force step-by-step reasoning ---
structured_cot_prompt = f"""Solve the following problem. 
First break the problem into reasoning steps, then compute the answer.

Output format:

Reasoning:
<step-by-step reasoning>

Answer:
<final answer>

Problem:

{tennis_problem}"""

print("=== Structured CoT ===\n")
r_cot = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": structured_cot_prompt}],
    temperature=0,
)
print(r_cot.choices[0].message.content)

## Needle in a Haystack

Can the model find a single specific fact buried in a massive document?

We take the full text of *The Adventures of Sherlock Holmes* (~12,000 lines) with one small detail planted inside it, and ask the model to find it. This tests **long-context retrieval** — the ability to locate and extract a specific piece of information from a large input.

The "needle" is a modified line in the original text. Let's see if the model can find it.

In [ ]:
from pathlib import Path
import time

# Load the full Sherlock Holmes text (~12,000 lines, ~590 KB)
haystack_path = Path("labs/02_standalone_agents/prompting/sherlock.txt")
if not haystack_path.exists():
    haystack_path = Path("prompting/sherlock.txt")

haystack = haystack_path.read_text()
print(f"Loaded {len(haystack):,} characters ({len(haystack.splitlines()):,} lines)\n")

question = "Which building was depicted in the photo?"

start = time.time()
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer the question based only on the provided text."},
        {"role": "user", "content": f"{haystack}\n\n---\nQuestion: {question}"},
    ],
    temperature=0,
)
elapsed = time.time() - start

print(f"Question: {question}")
print(f"Answer:   {r.choices[0].message.content}")
print(f"\nTokens — input: {r.usage.prompt_tokens:,}, output: {r.usage.completion_tokens}")
print(f"Elapsed: {elapsed:.2f}s")

## Prompt Templates with Jinja2

So far we've written prompts as Python strings. That works for quick experiments, but it gets messy fast:
- **Readability** — long prompts with f-string interpolation are hard to review
- **Separation of concerns** — the prompt *wording* is a design decision; the Python code is plumbing
- **Reuse** — the same template can be rendered with different variables (different stories, different criteria)
- **Version control** — changes to the prompt show up as clean diffs in `.j2` files, not buried inside code

[Jinja2](https://jinja.palletsprojects.com/) is a templating engine. You write a text file with `{{ variable }}` placeholders, load it in Python, and call `.render()` with the actual values.

In [ ]:
from jinja2 import Template
from pathlib import Path

# Load the template from file
template = Template(Path("prompting/example_template.j2").read_text())

# Render it with concrete values
rendered = template.render(
    role="literary critic",
    tasks=["Identify the tone", "List key themes", "Rate writing quality 1-5"],
    output_format="json",
    text="It was the best of times, it was the worst of times...",
)

print("=== Rendered prompt ===")
print(rendered)

### Jinja2 cheat sheet for prompts

| Syntax | What it does | Example |
|---|---|---|
| `{{ var }}` | Insert a variable | `{{ story_text }}` |
| `{% for x in list %}` | Loop over items | `{% for task in tasks %}` |
| `{% if cond %}` | Conditional block | `{% if output_format == "json" %}` |
| `{# comment #}` | Comment (not rendered) | `{# TODO: add few-shot examples #}` |
| `{{ var \| upper }}` | Filters (transform values) | `{{ name \| title }}` |

**Workflow summary:**
1. Write your prompt in a `.j2` file (keep it in a `prompting/` folder)
2. Load it with `Template(Path("prompting/my_prompt.j2").read_text())`
3. Render with `template.render(var1=value1, var2=value2)`
4. Pass the rendered string to the LLM as the message content